# Using Tools with Language Models

This notebook builds and tests a Python boundary between a proposed tool call and an executed function. We look up workshop supplies, calculate quotes, reject invalid requests, and record evidence for answers.

## Step 1 — Inspect the Workshop Data

Build a small, inspectable source of truth before connecting any model.

In [ ]:
import json

catalog = {
    "gloves": {"stock": 7, "unit_cents": 325},
    "goggles": {"stock": 4, "unit_cents": 850},
}
for item, fields in catalog.items():
    print(item, fields)
print("Items:", len(catalog))

**Q: If someone asks about current stock, why is a plausible answer from memory weaker evidence than a successful lookup?**

Memory can be outdated or wrong — it's based on what was true at some earlier point, not right now. A lookup hits the actual data source in real time, so the number it returns reflects the current state. Inventory changes constantly, so only a live query counts as reliable evidence.

## Step 2 — Write Two Narrow Tools

Separate retrieving stock from calculating a supply cost.

In [ ]:
def stock_lookup(item):
    """Return available units for a catalog item."""
    return {"item": item, "available": catalog[item]["stock"]}

def supply_quote(item, units):
    """Calculate cents for 1 through 20 units; reserve nothing."""
    total = catalog[item]["unit_cents"] * units
    return {"item": item, "units": units, "total_cents": total}

print(stock_lookup("gloves"))
print(supply_quote("gloves", 6))

**Q: Which tool supplies evidence for availability? Which supplies evidence for cost?**

`stock_lookup` gives availability — it returns how many units are in stock. `supply_quote` gives cost — it multiplies unit price by quantity. A quote says nothing about whether items are available, and a stock check says nothing about price.

## Step 3 — Describe the Inputs with Schemas

Give callers precise names, descriptions, types, and constraints using JSON Schema.

In [ ]:
item_rule = {"type": "string", "enum": list(catalog)}
stock_schema = {
    "type": "object",
    "properties": {"item": item_rule},
    "required": ["item"], "additionalProperties": False,
}
quote_schema = {
    "type": "object",
    "properties": {
        "item": item_rule,
        "units": {"type": "integer", "minimum": 1, "maximum": 20},
    },
    "required": ["item", "units"], "additionalProperties": False,
}
print(json.dumps(quote_schema, indent=2))

**Q: Does writing `maximum: 20` in a dictionary prevent a Python function from receiving 21? Explain where enforcement must happen.**

No. The schema is just a dictionary describing what's allowed — Python doesn't read it and block bad values on its own. `supply_quote` will accept `units=21` if called directly. Enforcement has to happen in a separate validation step between receiving the call and running the function. The schema describes the contract; your code enforces it.

## Step 4 — Create a Registry and a Prompt

Connect public tool names to approved functions and share useful definitions.

In [ ]:
registry = {
    "stock_lookup": {"run": stock_lookup,
        "description": "Read available workshop units for one item.",
        "schema": stock_schema},
    "supply_quote": {"run": supply_quote,
        "description": "Calculate a cost in cents without reserving stock.",
        "schema": quote_schema},
}
definitions = [
    {"name": name, "description": spec["description"],
     "input_schema": spec["schema"]}
    for name, spec in registry.items()
]
prompt = ('For a tool request return JSON with name and arguments. '
          'For a direct answer return JSON with answer. Available tools: '
          + json.dumps(definitions))
print([tool["name"] for tool in definitions])

**Q: Improve the description of `supply_quote` so a caller will not confuse a quote with an order.**

Better description: *"Calculate a cost estimate in cents for a given quantity. This is a price calculation only — it does not reserve stock, place an order, or guarantee availability."* This makes it clear the tool just returns a number, not a commitment.

## Step 5 — Parse and Validate a Proposed Call

Reject malformed or out-of-contract requests before dispatch.

In [ ]:
def parse_call(text):
    call = json.loads(text)
    if not isinstance(call, dict) or set(call) != {"name", "arguments"}:
        raise ValueError("Expected name and arguments only")
    name, args = call["name"], call["arguments"]
    if not isinstance(name, str) or name not in registry:
        raise ValueError("Unknown tool")
    schema = registry[name]["schema"]
    if not isinstance(args, dict) or set(args) != set(schema["required"]):
        raise ValueError("Missing or extra argument")
    for key, rule in schema["properties"].items():
        value = args[key]
        if rule["type"] == "string":
            if not isinstance(value, str) or value not in rule["enum"]:
                raise ValueError("Invalid item")
        elif rule["type"] == "integer":
            if type(value) is not int:
                raise ValueError("Quantity must be an integer")
            if not rule["minimum"] <= value <= rule["maximum"]:
                raise ValueError("Quantity outside allowed range")
    return call

proposal = '{"name":"supply_quote","arguments":{"item":"gloves","units":6}}'
print(parse_call(proposal))

**Q: What different questions do parsing and validation answer?**

Parsing asks: *"Is this well-formed data?"* Can the text be decoded into a Python object? If it's not valid JSON, parsing fails immediately.

Validation asks: *"Does this data meet the contract?"* Is the tool name known, are the right arguments present, and are the values the right types and in range? Something can be valid JSON and still fail validation.

## Step 6 — Execute Through a Controlled Boundary

Turn an accepted proposal into a structured result and enforce an action policy.

In [ ]:
drafts = []
def draft_request(item, units):
    draft = {"item": item, "units": units}
    drafts.append(draft)
    return {"draft_number": len(drafts), "status": "draft only"}

registry["draft_request"] = {
    "run": draft_request, "description": "Create a local supply draft.",
    "schema": quote_schema,
}
approval_tools = {"draft_request"}

def execute_call(text, approved=False):
    try:
        call = parse_call(text)
        if call["name"] in approval_tools and not approved:
            return {"ok": False, "error": "Approval required"}
        result = registry[call["name"]]["run"](**call["arguments"])
        return {"ok": True, "result": result}
    except (ValueError, TypeError, KeyError) as error:
        return {"ok": False, "error": str(error)}

print(execute_call(proposal))

**Q: Why must an approval decision stay outside a model-produced argument dictionary?**

Generated output is untrusted. If a caller could include `approved: true` in its own arguments, it could bypass the safety gate. Approval has to come from the host application or the user — a trusted source, not from the call itself. The caller proposes actions; the application authorizes them. Keeping those separate prevents a caller from granting itself permissions.

## Step 7 — Exercise Failures Before Adding a Model

Distinguish formatting errors, argument errors, unknown tools, and blocked actions.

In [ ]:
fixtures = [
    proposal,
    "not JSON",
    '{"name":"erase_all","arguments":{}}',
    '{"name":"supply_quote","arguments":{"item":"gloves","units":true}}',
    '{"name":"supply_quote","arguments":{"item":"gloves","units":21}}',
    '{"name":"draft_request","arguments":{"item":"gloves","units":6}}',
]
for text in fixtures:
    print(execute_call(text))
print("Drafts before approval:", len(drafts))
print(execute_call(fixtures[-1], approved=True))
print("Drafts after approval:", len(drafts))

**Q: A request can be valid JSON and still fail. Identify two such examples above.**

1. `{"name":"erase_all","arguments":{}}` — valid JSON, but `erase_all` isn't a registered tool, so validation rejects it.
2. `{"name":"supply_quote","arguments":{"item":"gloves","units":true}}` — valid JSON, but `true` is a boolean, not an integer. The validator uses `type(value) is int` which rejects booleans.

## Step 8 — Record a Complete Tool Exchange

Carry an observation back to an answer-producing stage.

In [ ]:
history = [{"role": "user", "content": "Quote six gloves."}]
history.append({"role": "assistant", "call_id": "quote-1",
                "content": proposal})
observation = execute_call(proposal)
history.append({"role": "tool", "call_id": "quote-1",
                "content": json.dumps(observation)})
if observation["ok"]:
    cents = observation["result"]["total_cents"]
    answer = f"Six gloves cost ${cents / 100:.2f}. Stock is unverified."
else:
    answer = "The quote failed; no price is confirmed."
history.append({"role": "assistant", "content": answer})
print(answer)
print([message["role"] for message in history])

**Q: What additional call would justify claiming that six gloves are available?**

A `stock_lookup` call for gloves. The quote tool only returns price — it says nothing about whether the items exist in stock. You'd need `stock_lookup("gloves")` returning an `available` count of at least 6 to make that claim.

## Step 9 — Normalize a Native Tool Response

Adapt a structured provider response to the same validated dispatcher.

In [ ]:
native_fixture = {
    "id": "stock-2", "type": "function",
    "function": {"name": "stock_lookup",
                 "arguments": '{"item":"gloves"}'},
}
def normalize_native(record):
    function = record["function"]
    args = function["arguments"]
    if isinstance(args, str):
        args = json.loads(args)
    return json.dumps({"name": function["name"], "arguments": args})

native_result = execute_call(normalize_native(native_fixture))
tool_reply = {"role": "tool", "tool_call_id": native_fixture["id"],
              "content": json.dumps(native_result)}
print(tool_reply)

**Q: Why should the prompt-based and native paths share the same execution boundary?**

Security shouldn't depend on how the call arrived. Whether it came from parsed text or a structured field, the same validation, authorization, and execution logic must apply. Two separate paths means two places for bugs. One boundary means one place to enforce the contract and one place to audit.

## Step 10 — Retrieve a Small Tool Shortlist

Show discovery separately from choosing and authorizing a call.

In [ ]:
def discover(query):
    words = set(query.lower().split())
    tags = {
        "stock_lookup": {"stock", "available", "inventory"},
        "supply_quote": {"cost", "price", "quote"},
        "draft_request": {"draft", "request"},
    }
    return [name for name, hints in tags.items() if words & hints]

print(discover("available gloves"))
print(discover("cost of gloves"))
print(discover("hello"))

**Q: What would you do if a shortlist is empty for a genuine inventory request?**

An empty shortlist means the keywords didn't match any tags. The fallback should broaden the search — try synonyms or show all available tools so the right one can still be selected. Never silently fail; either retry with a wider search or present the full catalog.

## Step 11 — Trace an MCP-Style Discovery Exchange

Separate host, client, and server responsibilities using an offline role play.

In [ ]:
def simulated_server(method, params):
    if method == "tools/list":
        return {"tools": [
            {"name": name, "description": spec["description"],
             "inputSchema": spec["schema"]}
            for name, spec in registry.items()
        ]}
    if method == "tools/call":
        return execute_call(json.dumps(params))
    return {"error": "Unsupported method in simulation"}

listed = simulated_server("tools/list", {})
print("Discovered:", [tool["name"] for tool in listed["tools"]])
params = {"name": "stock_lookup", "arguments": {"item": "goggles"}}
print(simulated_server("tools/call", params))
print("OFFLINE ROLE PLAY ONLY")

**Q: For three applications and four services, compare 12 custom pairings with 7 conceptual adapters. Why is this only a simplification of real integration work?**

Without a shared protocol: 3 × 4 = 12 custom integrations. With a shared protocol: 3 + 4 = 7 adapters.

It's a simplification because the protocol only standardizes the message format. You still need per-service authentication, version compatibility, error handling, transport security, and ongoing maintenance. The protocol reduces the format problem but doesn't eliminate the operational work.

## Step 12 — Choose Procedural Guidance Only When Needed

Distinguish a tool contract from a reusable workflow and its supporting resources.

In [ ]:
skill_demo = {
    "name": "workshop-supply-check",
    "description": "Use when checking a workshop supply request.",
    "instructions": (
        "Look up stock. Calculate the quote. Report both results. "
        "Ask the user before creating any draft."
    ),
    "resources": {"report-format.md": "Item, availability, cost, decision"},
}
print("Layer 1:", skill_demo["name"], skill_demo["description"])
print("Layer 2:", skill_demo["instructions"])
print("Layer 3:", skill_demo["resources"]["report-format.md"])
print("Instructions alone do not enforce authorization.")

**Q: Which layer would you expand to describe the expected student report? Which code boundary would you strengthen to enforce stock rules?**

Expand **Layer 3** (resources) to describe the report — that's where a detailed template with sections and required fields belongs.

Strengthen the **execution boundary** in `execute_call` to enforce stock rules. Right now `supply_quote` calculates cost even if quantity exceeds stock. A stock check before dispatch would catch that. Instructions (Layer 2) can suggest a workflow, but only code at the boundary actually prevents bad actions.

---

## How Models Learn Tool Use

**In-context learning:** Examples in the prompt show the desired call-and-response pattern. This changes the context, not the model weights. It uses up context space and can still miss on new wording.

**Supervised fine-tuning (SFT):** Training on examples of correct tool calls updates weights so the right behavior becomes more likely. Training data includes calls, results, answers, failures, and alternatives. Toolformer is an early approach: it inserts API calls into text, runs them, keeps the useful ones based on prediction loss, then trains on the result.

**Reinforcement learning (RL):** Updates a policy using rewards from attempted behavior. Rewards check tool name, arguments, format, or task success. ToolRL studies reward design using GRPO, comparing sampled attempts within a group. Neither SFT nor RL guarantees reliable tool use.

**Search during reasoning:** Interleaves computation with external calls. Search-R1 trains search interactions with outcome rewards and masks retrieved tokens from the loss — the search service produced those tokens, so the policy shouldn't train as if it generated them.

---

## Protocols and Procedural Guidance

**Integration:** N apps and M services need N×M custom integrations. A shared protocol like MCP reduces that toward N+M. But you still need per-service auth, versioning, error handling, and maintenance.

**The MCP exchange:** The host discovers a server's tools via a client, shares descriptions with the model, the model proposes a call, the host validates and routes it to the server, and the result enters the conversation. MCP uses JSON-RPC. Tools do operations, resources provide data, prompts offer reusable templates.

**Skills:** A Skill packages procedural guidance with progressive disclosure — name and description for selection, instructions when it applies, extra references when needed. A tool exposes a capability, MCP connects capabilities, a Skill explains a procedure. Loading instructions doesn't grant permission.

---

## Practice Exercises

### Exercise 1 — Availability and Cost

Determine whether the workshop has enough goggles for a request of 5, calculate the quote, and report the difference.

In [ ]:
stock_result = stock_lookup("goggles")
quote_result = supply_quote("goggles", 5)

available = stock_result["available"]
requested = 5
shortage = requested - available

print(f"Available: {available}")
print(f"Requested: {requested}")
print(f"Shortage: {shortage}")
print(f"Quote: {quote_result['total_cents']} cents (${quote_result['total_cents'] / 100:.2f})")
print()
print("A quote is a price calculation, not a fulfilled request.")
print("The workshop cannot fill this — 4 available but 5 requested.")
print("No draft created.")

### Exercise 2 — Boundary Checks

Test malformed requests and verify none produce side effects.

In [ ]:
drafts_before = len(drafts)

boundary_tests = [
    ("Missing units key",
     '{"name":"supply_quote","arguments":{"item":"gloves"}}'),
    ("Unexpected price key",
     '{"name":"supply_quote","arguments":{"item":"gloves","units":5,"price":100}}'),
    ("Item tape (not in catalog)",
     '{"name":"supply_quote","arguments":{"item":"tape","units":3}}'),
    ("Units 0 (below minimum)",
     '{"name":"supply_quote","arguments":{"item":"gloves","units":0}}'),
    ("Units 20 (at maximum — should succeed)",
     '{"name":"supply_quote","arguments":{"item":"gloves","units":20}}'),
    ("Units 21 (above maximum)",
     '{"name":"supply_quote","arguments":{"item":"gloves","units":21}}'),
    ('Units "6" (string, not integer)',
     '{"name":"supply_quote","arguments":{"item":"gloves","units":"6"}}'),
    ("Units true (boolean, not integer)",
     '{"name":"supply_quote","arguments":{"item":"gloves","units":true}}'),
]

for label, test in boundary_tests:
    result = execute_call(test)
    status = "OK" if result["ok"] else f"REJECTED: {result['error']}"
    print(f"{label:45s} -> {status}")

print(f"\nDrafts before: {drafts_before}, after: {len(drafts)} (no change = correct)")

### Exercise 3 — Descriptions and Selection

Write few-shot demonstrations and explain how ambiguity can cause errors.

In [ ]:
few_shot_examples = """
--- Few-shot: stock_lookup ---
User: How many goggles do we have?
Assistant: {"name": "stock_lookup", "arguments": {"item": "goggles"}}
Tool result: {"item": "goggles", "available": 4}
Assistant: {"answer": "There are 4 goggles available."}

--- Few-shot: supply_quote ---
User: What would 3 pairs of gloves cost?
Assistant: {"name": "supply_quote", "arguments": {"item": "gloves", "units": 3}}
Tool result: {"item": "gloves", "units": 3, "total_cents": 975}
Assistant: {"answer": "3 pairs of gloves cost $9.75. Quote only, not an order."}

--- Greeting (no tool needed) ---
User: Hello!
Assistant: {"answer": "Hello! I can check supply availability and calculate quotes."}
"""
print(few_shot_examples)

**How ambiguous descriptions and incomplete shortlists cause errors:**

If both tools had vague descriptions like "get item info," a caller could pick the wrong one — asking for price but getting stock. That's a selection error.

An incomplete shortlist is different. If the keyword filter only returns `supply_quote` for "do we have enough gloves," the caller never sees `stock_lookup`. It might use the wrong tool or give up. Different cause, same result: wrong tool or no tool when one was needed.

### Exercise 4 — A Complete Exchange

Extend the transcript with both a stock lookup and a quote for six gloves, using separate IDs.

In [ ]:
exchange = [{"role": "user", "content": "Can I get six gloves? Check stock and price."}]

stock_call = '{"name":"stock_lookup","arguments":{"item":"gloves"}}'
exchange.append({"role": "assistant", "call_id": "stock-1", "content": stock_call})
stock_obs = execute_call(stock_call)
exchange.append({"role": "tool", "call_id": "stock-1",
                 "content": json.dumps(stock_obs)})

quote_call = '{"name":"supply_quote","arguments":{"item":"gloves","units":6}}'
exchange.append({"role": "assistant", "call_id": "quote-2", "content": quote_call})
quote_obs = execute_call(quote_call)
exchange.append({"role": "tool", "call_id": "quote-2",
                 "content": json.dumps(quote_obs)})

avail = stock_obs["result"]["available"]
cost = quote_obs["result"]["total_cents"]
final = (f"There are {avail} gloves available, covering the request of 6. "
         f"Cost: ${cost / 100:.2f}. This is a quote — no stock reserved.")
exchange.append({"role": "assistant", "content": final})

print(final)
print("Roles:", [m["role"] for m in exchange])

**Challenge: Bounded scripted loop**

In [ ]:
CALL_BUDGET = 3

scripted_calls = [
    ("stock-a", '{"name":"stock_lookup","arguments":{"item":"gloves"}}'),
    ("quote-b", '{"name":"supply_quote","arguments":{"item":"gloves","units":6}}'),
]

loop_history = [{"role": "user", "content": "Check stock and quote for 6 gloves."}]
calls_made = 0

for call_id, call_text in scripted_calls:
    if calls_made >= CALL_BUDGET:
        loop_history.append({"role": "assistant",
                             "content": "Call budget exhausted. Stopping."})
        break
    loop_history.append({"role": "assistant", "call_id": call_id, "content": call_text})
    obs = execute_call(call_text)
    loop_history.append({"role": "tool", "call_id": call_id,
                         "content": json.dumps(obs)})
    calls_made += 1

tool_results = [json.loads(m["content"]) for m in loop_history if m["role"] == "tool"]
stock_ev = next(r["result"] for r in tool_results if "available" in r.get("result", {}))
quote_ev = next(r["result"] for r in tool_results if "total_cents" in r.get("result", {}))

final_answer = (f"{stock_ev['available']} gloves available. "
                f"Quote: ${quote_ev['total_cents'] / 100:.2f}. No reservation made.")
loop_history.append({"role": "assistant", "content": final_answer})

print(f"Calls made: {calls_made}/{CALL_BUDGET}")
print(f"Answer: {final_answer}")
print("Roles:", [m["role"] for m in loop_history])

### Exercise 5 — Workflow and Protocol

**MCP Architecture Sketch:**

```
┌─────────────────────────────────────────┐
│                HOST                      │
│  ┌────────┐  ┌───────┐  ┌────────────┐  │
│  │  User  │◄►│ Model │  │  Policy    │  │
│  └────────┘  └───┬───┘  │ (consent,  │  │
│                  │      │  authz)    │  │
│            ┌─────▼────┐ └─────┬──────┘  │
│            │  CLIENT  │◄──────┘          │
│            └─────┬────┘                  │
└──────────────────┼───────────────────────┘
                   │ JSON-RPC
            ┌──────▼───────┐
            │   SERVER     │
            │              │
            │ tools/list   │  ← Discovery
            │ tools/call   │  ← Execution
            └──────────────┘
```

**Discovery** goes server → client: the client asks `tools/list`, the server returns tool names, descriptions, and schemas.

**Execution** goes client → server: after the host validates policy, the client sends `tools/call` and gets the result back.

**Workshop Supply Skill draft:**

```yaml
# SKILL.md
---
name: workshop-supply-check
description: Use when checking a workshop supply request.
---

1. Use stock_lookup to check available units.
2. Use supply_quote to calculate cost.
3. Report both using references/report-format.md.
4. If stock is short, state the shortage.
5. Ask the user before creating any draft.
```

**Referenced resource** (`references/report-format.md`):
```
## Supply Check Report
- Item: [name]
- Requested: [quantity]
- Available: [stock count]
- Status: [sufficient / shortage of N]
- Quote: $[amount]
- Decision: [awaiting approval / not applicable]
```

**What the host must enforce even if a Skill asks for an action:** The host enforces its authorization policy no matter what instructions say. A Skill can suggest "create a draft," but the host still requires explicit user approval before running write tools. The Skill can't bypass the `approved` flag in `execute_call` — that gate belongs to the application.